## Notebook 6 - Feature Engineering 

This notebook runs **after the final Cleaning notebook** 

### This notebook includes
1. Loading the final cleaned structured dataset.
2. Removal of all the columns that we decided in the cleaning stage.
3. Creation and export of a fixed stratified Train/Test split before generating the supervised NLP-derived feature, so the same held-out Test set is preserved throughout the downstream modeling pipeline.
4. Leakage-safe generation of the TF-IDF + Logistic Regression NLP risk score.
5. Deterministic clinical feature engineering.
6. ICD9-based grouped feature infrastructure.
7. Audit of redundant textual `DIAGNOSIS` / `PROCEDURES` fields that describe the same ICD9 information.
8. Medication-text review and clinically-approved medication-family flags.
9. Creation of `transferred_to_icu_directly` from `icu_transfer_start_date` relative to calculated surgery end.
10. Surgeon/anesthesiologist cardinality and frequency audit — **the codes are not dropped here**.
11. Removal from the final tabular feature table of raw NLP narrative text and confirmed redundant raw source fields **only after their derived features have been created**.
12. Saving one **FINAL FEATURE TABLE** for the next notebook.



## Step 0 - Setup and project configuration


In [8]:
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import Counter
import re
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_colwidth", 200)

TARGET_COL = "target_binary"
CASE_ID_COL = "case_number"

CLEANED_DATA_PATH = Path(r"H:\27 project\cleaning\cleaning_outputs\02_cleaned_data\df_zihum_clean.csv")


OUTPUT_DIR = Path(r"H:\27 project\feature_engineering\fe_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Event-based, matching the Cleaning notebook: pre-op through end of surgery, no
# extended post-surgery monitoring window. Kept at 0 only for the printed status
# line in Step 1 - not a real clock offset from admission.
PREDICTION_WINDOW_HOURS = 0

# ------------------------------------------------------------------
# NLP MERGE
# ------------------------------------------------------------------
USE_NLP_FEATURES = True
NLP_FEATURE_PATH = Path(r"H:\27 project\feature_engineering\fe_outputs\tfidf_lr_nlp_score.csv" )

NLP_ID_COL = "case_number"
NLP_RISK_SCORE_COL = "risk_score"

# These are the original free-text narrative columns used by the NLP pipeline.
# They are NOT intended as direct predictors in the final tabular model.
NLP_RAW_TEXT_COLS = [
    "first_hospitalization_disease_history",
    "description_surgical_procedure",
    "first_hospitalization_main_complaint" # not in nlp but the data in here is in the disease_history column
]

# These 3 date columns are used transiently in Step 3 (to compute transferred_to_icu_directly - see below) 
# and then dropped, unconditionally, once that computation is done. There is no scenario where a raw 
# admission/surgery/discharge timestamp should be fed to the model directly (data leakage risk: Step 6
# - Pre-Modeling - would otherwise One-Hot-encode it as a near-unique categorical column). discharge_date 
# is not currently used by any feature here, but stays in this list for the same reason - 
# a raw timestamp is never a safe predictor.
DATE_SOURCE_COLS_TO_DROP = [
    "admission_date",
    "activity_date",
    "icu_transfer_start_date",
    "surgery_bed_transfer_start_date",
    "surgey_bed_transfer_end_date",
    "recovery_bed_transfer_start_date",
    "recovery_bed_transfer_end_date",
    "CVC_start_date",
    "catheter_start_date",
    "added1_drain_left_in_place_start_date"
]

DERIVED_FEATURE_RAW_COLS_TO_DROP = [
    "past_surgery_first_hospitalization",
    "rpreop_bmi_value",
    "preop_regular_medications",
    "preop_mental_health",
    "catheter_route",
    "catheter_order_dose_remarks",
    "added1_drain_left_in_place_clean_from_surgery",
    
]

# ------------------------------------------------------------------
# ICU TRANSFER (replaces the removed discharged_within_24h - see Step 3)
# ------------------------------------------------------------------
# icu_transfer_start_date / icu_bed are raw source columns, never fed to the model
# directly (same reasoning as DATE_SOURCE_COLS_TO_DROP) - dropped unconditionally in
# Step 7 once transferred_to_icu_directly is computed from them in Step 3.
ICU_TRANSFER_DATE_COL = "icu_transfer_start_date"
ICU_BED_COL = "icu_bed"
ICU_RAW_COLS_TO_DROP = [ICU_TRANSFER_DATE_COL, ICU_BED_COL]

# Reuses the Cleaning notebook's TOLERANCE_HOURS_AFTER_SURGERY convention/default -
# update manually if that value changes there, this is a separate copy.
ICU_TOLERANCE_HOURS_AFTER_SURGERY: float = 6.0
SURGERY_LENGTH_COL = "added1_length_of_surgery"

# ------------------------------------------------------------------
# ICD9 / DIAGNOSIS / PROCEDURES — OFFLINE JSON, NO EXTERNAL LIBRARY
# ------------------------------------------------------------------
ICD9_DICTIONARY_PATH = Path(r"H:\27 project\icd9_surgical_dictionary.json")

ICD9_DIAGNOSIS_SOURCE_COLUMNS = [
    "diagnosis_icd9_1",
    "diagnosis_icd9_2",
    "diagnosis_icd9_3",
    "diagnosis_icd9_4",
    "diagnosis_icd9_5",
]

ICD9_PROCEDURE_SOURCE_COLUMNS = [
    "procedure_icd9_1",
    "procedure_icd9_2",
    "procedure_icd9_3",
    "procedure_icd9_4",
    "procedure_icd9_5",
]

DIAGNOSIS_TEXT_COLUMNS = [
    "diagnosis_1", "diagnosis_2", "diagnosis_3", "diagnosis_4", "diagnosis_5",
]
PROCEDURE_TEXT_COLUMNS = [
    "procedure_1", "procedure_2", "procedure_3", "procedure_4", "procedure_5",
]

# Group flags are now created automatically from the supplied offline dictionary.
ENABLE_COMORBIDITY_FLAGS = True
ENABLE_PROCEDURE_GROUP_FLAGS = True

# Exact redundant source fields are kept as reference and can be removed after
# successful feature extraction.
CONFIRMED_REDUNDANT_DIAGNOSIS_PROCEDURE_TEXT_COLS: List[str] = [
    "diagnosis_2", "diagnosis_3", "diagnosis_4", "diagnosis_5",
    "procedure_2", "procedure_3", "procedure_4", "procedure_5",
]

# ------------------------------------------------------------------
# PREOPERATIVE REGULAR MEDICATIONS
# ------------------------------------------------------------------
DRUG_SOURCE_COLUMN = "preop_regular_medications"

# Clinically approved medication-family patterns go here.
# Each family becomes a YES/NO feature if enabled.
ENABLE_DRUG_FLAGS = True
DRUG_FAMILY_PATTERNS: Dict[str, str] = {
    # will be fill later in this notebook.
    # FORMAT EXAMPLES ONLY 
    # "takes_PPI": r"\b(?:NEXIUM|OMEPRADEX|OMEPRAZOLE|PANTOPRAZOLE)\b",
    # "takes_statin": r"\b(?:LIPITOR|ATORVASTATIN|ROSUVASTATIN)\b",
}

# Tokens that are common instructions/doses rather than medication names.
MEDICATION_STOPWORDS = {
    "TAB", "TABLET", "CAP", "CAPSULE", "MG", "ML", "DAY", "DAILY", "MORN",
    "MORNING", "EVENING", "NIGHT", "STOPPED", "INHALER", "SOS", "MONTH",
    "PO", "IV", "X", "TAKE", "TIMES", "WEEK", "AM", "PM", "ACID", "EVEN"
    "בוקר", "ערב", "לילה", "פעם", "ביום", "כדור", "טבליה", "טבליות",
    "טיפול", "קבוע", "תרופתי", "לא", "זריקה", "בחודש", "ללא", "בקר",
    "לפי", "וערב", "הצורך", "בשבוע", "PLUS", "נוטל", "הופסק", "צהריים", 
    "בבוקר", "NEW", "אחת", "ONE", "צורך", "בערב", "לפני", "יום", "צהרים", 
    "רציף", "EYE", "פעמיים", "לשבוע", "ימים", "טיפות", "פעמים", "רפואי", 
    "TEVA", "לסירוגין", "עין", "טיפה", "השינה", "NON", "בלילה", "עיניים", 
    "מקג", "בשבועיים", "כאבים", "שנה", "שבועות", "נגד", "ראשון", "ביומיים",
    "הפסיק", "UNKNOWN", "בוקר"
}


# ------------------------------------------------------------------
# EDA-INFORMED DETERMINISTIC FEATURE ENGINEERING
# ------------------------------------------------------------------
# These features were selected after EDA because they capture clinically interpretable
# dimensions of baseline risk / surgical complexity without creating many redundant
# variants of diagnosis/procedure counts.
BMI_SOURCE_CANDIDATES = ["preop_bmi_value"]
ASA_SOURCE_CANDIDATES = ["preop_asa_value"]
DRAIN_PRESENCE_CANDIDATES = ["added1_drain_left_in_place_clean_from_surgery"]

# ------------------------------------------------------------------
# SURGEON / ANESTHESIOLOGIST
# These are audited, not automatically removed.
# ------------------------------------------------------------------
CLINICIAN_CODE_COLS = [
    "surgeon_code",
    "anasthesiologist_code",
]


In [9]:
# OFFLINE ICD-9-CM DICTIONARY
ICD9_DICTIONARY_PATH = Path(r"H:\27 project\icd9_surgical_dictionary.json")


def load_icd9_dictionary(path: Path = ICD9_DICTIONARY_PATH) -> Dict:
    """
    INFO:
    Fails soft: returns an empty dictionary (both groups empty) instead of raising,
    so a missing/corrupt file does not crash the rest of the notebook - Step 4 will
    simply report "no source columns found" / create nothing, same failure mode as
    every other optional feature block here (NLP merge, medication flags, etc.).
    """
    try:
        if not path.exists():
            print(f"WARNING: ICD9 dictionary not found at {path} - "
                  "diagnosis/procedure grouped features will be skipped.")
            return {"diagnosis_groups": {}, "procedure_groups": {}}
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError) as exc:
        print(f"WARNING: could not load ICD9 dictionary ({exc}) - "
              "diagnosis/procedure grouped features will be skipped.")
        return {"diagnosis_groups": {}, "procedure_groups": {}}


def normalize_icd9_code(value, kind: str) -> str:
    """
    Normalize ICD-9 codes while preserving leading-zero semantics.
    - diagnosis numeric codes use 3 digits before the decimal when needed
      (e.g. 38.9 -> 038.9)
    - procedure codes use 2 digits before the decimal when needed
      (e.g. 3.09 -> 03.09)
    V/E diagnosis codes are preserved.
    """
    if pd.isna(value):
        return ""

    s = str(value).strip().upper()
    if not s or s in {"NAN", "NONE", "<NA>"}:
        return ""

    # Remove trailing .0 introduced by Excel/pandas only when it represents an integer.
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]

    s = s.replace(" ", "")

    if s.startswith(("V", "E")):
        return s

    if re.fullmatch(r"\d+(?:\.\d+)?", s):
        left, dot, right = s.partition(".")
        width = 3 if kind == "diagnosis" else 2
        left = left.zfill(width)
        return left + (dot + right if dot else "")

    return s


def normalize_clinical_text(value) -> str:
    if pd.isna(value):
        return ""
    s = str(value).lower().strip()
    s = re.sub(r"[\t\r\n]+", " ", s)
    s = re.sub(r"[^0-9a-zA-Z\u0590-\u05FF]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


## Step 1 - Load the cleaned dataset

At this stage the data has already passed the Cleaning notebook.  
The checks below are therefore **handoff checks**, not a second Cleaning process.


In [10]:
df_final = pd.read_csv(CLEANED_DATA_PATH)
df_final = df_final.copy()

print(f"Loaded cleaned dataset: {df_final.shape}")

if TARGET_COL not in df_final.columns:
    raise KeyError(f"Missing required target column: {TARGET_COL}")

if CASE_ID_COL not in df_final.columns:
    raise KeyError(f"Missing required surgical-case identifier: {CASE_ID_COL}")

if df_final[CASE_ID_COL].isna().any():
    raise ValueError(f"{CASE_ID_COL} contains missing identifiers.")

print("Prediction window: pre-op through end of surgery (event-based, no post-surgery monitoring window)")
print("Feature Engineering starts from the cleaned dataset without Train/Test splitting.")


Loaded cleaned dataset: (31849, 97)
Prediction window: pre-op through end of surgery (event-based, no post-surgery monitoring window)
Feature Engineering starts from the cleaned dataset without Train/Test splitting.


C:\Users\mtal\AppData\Local\Temp\2\ipykernel_8476\2171791766.py:1: DtypeWarning: Columns (38,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv(CLEANED_DATA_PATH)


## Step 2 - Leakage-safe NLP-risk score generation

### Risk score from the TF-IDF + Logistic Regression model

Creates and saves the fixed Train/Test split so the PreModeling notebook uses the exact same split.

Uses the winning NLP model (TF-IDF = Logistic Regression) to generate leakage-safe risk scores.

Train scores are generated out-of-fold, while Test scores are generated by a model fitted only on the full Train set.


In [11]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

SCORE_TFIDF_PATH = r"H:\27 project\feature_engineering\fe_outputs\tfidf_lr_nlp_score.csv" 
SPLIT_PATH = r"H:\27 project\feature_engineering\fe_outputs\locked_train_test_split.csv"

TEST_SIZE = 0.3
RANDOM_STATE = 42
N_SPLITS = 5

CASE_ID_COL = "case_number"
TARGET_COL = "target_binary"
text_col_1, text_col_2 =   "first_hospitalization_disease_history", "description_surgical_procedure",

# Lock the final train-test split
train_idx, test_idx = train_test_split(
    np.arange(len(df_final)), 
    test_size=TEST_SIZE, 
    random_state=42, 
    stratify=df_final[TARGET_COL])

train_mask = np.zeros(len(df_final), dtype=bool)
test_mask = np.zeros(len(df_final), dtype=bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

split_export = pd.DataFrame({CASE_ID_COL: df_final[CASE_ID_COL].values, 
                             "split": np.where(train_mask, "train", "test")
                             }) 
split_export.to_csv(SPLIT_PATH, index=False, encoding="utf-8-sig")
print("Locked split saved:", split_export["split"].value_counts().to_dict())

# prepare output column
risk_score = np.full(len(df_final), np.nan, dtype=float)
y_train_full = (df_final.loc[train_mask, TARGET_COL].astype(int).values)

# train: generate out-of-fold nlp scores
## each training case is scored by a tf-idf + Logistic regression model that was NOT trained on that case.
## TF-IDF is also fitted separately inside each fold, preventing vocabulary/IDF information from the validation
## fold from entering the coresponding fold model.
train_positions = np.where(train_mask)[0]
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True,random_state=RANDOM_STATE)

for fold, (fit_rel_idx, val_rel_idx) in enumerate(skf.split(train_positions, y_train_full), start=1):
    fit_idx = train_positions[fit_rel_idx]
    val_idx = train_positions[val_rel_idx]

    # separate vectorizers for the two twxt fields
    tfidf_1_fold = TfidfVectorizer()  
    tfidf_2_fold = TfidfVectorizer()

    X1_fit = tfidf_1_fold.fit_transform(df_final.loc[fit_idx][text_col_1].fillna("").astype(str))
    X2_fit = tfidf_2_fold.fit_transform(df_final.loc[fit_idx][text_col_2].fillna("").astype(str))
    X_fit = hstack([X1_fit, X2_fit])

    X1_val = tfidf_1_fold.transform(df_final.loc[val_idx][text_col_1].fillna("").astype(str))
    X2_val = tfidf_2_fold.transform(df_final.loc[val_idx][text_col_2].fillna("").astype(str))
    X_val = hstack([X1_val, X2_val])

    y_fit = (df_final.loc[fit_idx, TARGET_COL].astype(int).values)
    fold_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    fold_model.fit(X_fit, y_fit)

    risk_score[val_idx] = (fold_model.predict_proba(X_val)[:, 1])
    print(f"Fold {fold}/{N_SPLITS} completed:"
          f"{len(val_idx)} training cases scored")

Locked split saved: {'train': 22294, 'test': 9555}
Fold 1/5 completed:4459 training cases scored
Fold 2/5 completed:4459 training cases scored
Fold 3/5 completed:4459 training cases scored
Fold 4/5 completed:4459 training cases scored
Fold 5/5 completed:4458 training cases scored


In [12]:
# TEST: fit final nlp model on all training data
# the held-out Test set is scored only after fitting the complete TF-IDF + Logistic Regression pipeline on the full train set
tfidf_1_final = TfidfVectorizer()  
tfidf_2_final = TfidfVectorizer()

X1_train_full = tfidf_1_final.fit_transform(df_final.loc[train_mask, text_col_1].fillna("").astype(str))
X2_train_full = tfidf_2_final.fit_transform(df_final.loc[train_mask, text_col_2].fillna("").astype(str))
X_train_full = hstack([X1_train_full, X2_train_full])

final_nlp_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
final_nlp_model.fit(X_train_full, y_train_full)

X1_test = tfidf_1_final.transform(df_final.loc[test_mask,text_col_1].fillna("").astype(str))
X2_test = tfidf_2_final.transform(df_final.loc[test_mask,text_col_2].fillna("").astype(str))
X_test = hstack([X1_test, X2_test])

risk_score[test_mask] = (final_nlp_model.predict_proba(X_test)[:, 1])

# Attach the NLP score
df_final["nlp_tfidf_logreg_risk_score"] = risk_score

In [14]:
# validation
assert df_final["nlp_tfidf_logreg_risk_score"].notna().all(), ("Some NLP risk scores were not generated.")
assert set(split_export["split"].unique()) == {"train", "test"}
print("Nlp risk score generated for all cases.")
print(df_final["nlp_tfidf_logreg_risk_score"].describe())

# export nlp score
df_final[[CASE_ID_COL, "nlp_tfidf_logreg_risk_score"]].to_csv(SCORE_TFIDF_PATH, index=False, encoding="utf-8-sig")
print(f"Saved NLP risk score for {len(df_final)} cases.")

Nlp risk score generated for all cases.
count    31849.000000
mean         0.076424
std          0.159751
min          0.000032
25%          0.005491
50%          0.014575
75%          0.049548
max          0.951326
Name: nlp_tfidf_logreg_risk_score, dtype: float64
Saved NLP risk score for 31849 cases.


## Step 3 - Create `transferred_to_icu_directly`

This feature represents the clinical decision discussed with the project mentors: whether the patient was transferred directly from surgery/recovery to ICU.

Clinical documentation rule confirmed by the team:
- `1`- ICU transfer recorded within the configured tolerance after surgery end.
- `0`- no direct ICU transfer. This includes cases with **no ICU transfer record**, because in this hospital clinicians document ICU transfer when it occurs; absence of documentation therefore means the patient did not enter ICU.

This is a clinically-defined encoding of missingness, not statistical imputation.


In [15]:
###### TRY

icu_start = pd.to_datetime(df_final["icu_transfer_start_date"], errors="coerce")
surgery_end = pd.to_datetime(df_final["surgey_bed_transfer_end_date"], errors = "coerce")

diff_minutes = (icu_start - surgery_end).dt.total_seconds() / 60.0
is_direct_transfer = (diff_minutes >= 0) & (diff_minutes <= 15)
df_final["transferred_to_icu_directly"] = is_direct_transfer.fillna(False).astype(int)


## EDA-informed clinical features

The features below were selected from patterns observed during the exploratory analysis of patients who developed infection, while avoiding target-specific or cohort-specific rules.

They summarize broad, clinically interpretable dimensions:
- BMI category (`bmi_category`);
- Payment category derived from the original `payment_by` values;
- higher preoperative physiological burden (`high_asa`, ASA ≥ 3);
- binary mental-health issue flag derived from the `mental_health` field;
- prior surgery flag derived from `past_surgery_fist_hospitalization`.
- preoperative invasive devices (`has_cvc`, `has_catheter`, `has_drain`, `invasive_device_count`).
- surgery-bed duration
- recovery-bed duration

`age_65_plus` is deliberately **not** created because cohort reduction later keeps patients aged 28 and above, and the original continuous age variable is retained.  
`received_preop_antibiotic` and `preop_antibiotic_type` are also deliberately **not** created in this notebook.

No cohort reduction or undersampling is performed in this notebook. Cohort reduction is treated as a sampling strategy and is applied only after the Train/Test split in Step 6.


In [16]:
def add_transferred_to_icu_feature(
    df_input: pd.DataFrame,
    icu_date_col: str = ICU_TRANSFER_DATE_COL,
    surgery_col: str = "activity_date",
    length_col: str = SURGERY_LENGTH_COL,
    tolerance_hours: float = ICU_TOLERANCE_HOURS_AFTER_SURGERY,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    INFO:
    transferred_to_icu_directly = 1 only when icu_transfer_start_date falls ON OR AFTER the calculated end of surgery 
    (activity_date + added1_length_of_surgery,minutes; missing length falls back to 0 - conservative, not lenient) AND within
    tolerance_hours after it. = 0 otherwise - including an ICU transfer recorded BEFORE surgery end (implausible/unrelated, 
    not a direct post-op transfer), an ICU transfer recorded too long after surgery end, or no ICU-transfer record at all. 
    The audit below reports these three 0-cases separately even though the
    stored feature value is the same (0) for all of them. length_col missing entirely (not just missing per-row) is handled
    the same way as the Cleaning notebook's timing gate: treated as 0 minutes duration.
    """
    result = df_input.copy()
    feature_col = "transferred_to_icu_directly"

    missing_cols = [c for c in [icu_date_col, surgery_col] if c not in result.columns]
    if missing_cols:
        return result, pd.DataFrame([{
            "feature": feature_col, "status": "source_columns_not_found",
            "missing_columns": missing_cols,
        }])

    icu_dt = pd.to_datetime(result[icu_date_col], errors="coerce")
    surgery_dt = pd.to_datetime(result[surgery_col], errors="coerce")

    if length_col in result.columns:
        length_minutes = pd.to_numeric(result[length_col], errors="coerce").fillna(0)
    else:
        length_minutes = pd.Series(0, index=result.index)

    surgery_end = surgery_dt + pd.to_timedelta(length_minutes, unit="m")
    deadline = surgery_end + pd.Timedelta(hours=tolerance_hours)

    has_icu_transfer = icu_dt.notna()
    within_window = (
        has_icu_transfer
        & surgery_end.notna()
        & (icu_dt >= surgery_end)
        & (icu_dt <= deadline)
    )

    feature = pd.Series(0, index=result.index, dtype="Int64")
    feature.loc[within_window] = 1
    result[feature_col] = feature

    audit = pd.DataFrame([{
        "feature": feature_col, "status": "created",
        "n_direct_transfer_1": int((feature == 1).sum()),
        "n_icu_but_not_direct_0": int((has_icu_transfer & ~within_window).sum()),
        "n_no_icu_transfer_recorded_encoded_0": int((~has_icu_transfer).sum()),
        "tolerance_hours_after_surgery": tolerance_hours,
    }])
    return result, audit


df_final, icu_transfer_audit = add_transferred_to_icu_feature(df_final)
display(icu_transfer_audit)


,feature,status,n_direct_transfer_1,n_icu_but_not_direct_0,n_no_icu_transfer_recorded_encoded_0,tolerance_hours_after_surgery
0,transferred_to_icu_directly,created,2411,247,29191,6.0


In [17]:
def resolve_first_existing_column(df_input: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in df_input.columns:
            return col
    return None


eda_feature_rows = []

# ----------------------------------------------------------
# payment_by category
# WHO-style broad categories are used as an interpretable representation;
# ----------------------------------------------------------
s = df_final["payment _by"].astype(str)
conds = [
    s.str.startswith("ביטוח חול", na=False),
    s.str.contains("תיירות מרפא", na=False),
    s.str.startswith("ביטוח", na=False),
    s.str.startswith("בית חולים", na=False) | s.str.contains("אסותא אשדוד", na=False),
    s.str.contains("כללית", na=False),
    s.str.contains("מאוחדת", na=False),
    s.str.contains("מכבי", na=False),
    s.str.contains("לאומית", na=False),
    s.str.contains("מטופל פרטי", na=False),
]
labels = ["foreign_insurance",
          "medical_tourism",
          "private_insurance",
          "public_hospital",
          "clalit", 
          "meuhedet",
          "maccabi",
          "leumit",
          "private_patient"]
df_final["payment_category"] = np.select(conds, labels, default="missing")




# ----------------------------------------------------------
# BMI category
# WHO-style broad categories are used as an interpretable representation;
# the original continuous BMI is retained as well.
# ----------------------------------------------------------
bmi_col = resolve_first_existing_column(df_final, BMI_SOURCE_CANDIDATES)
if bmi_col:
    bmi = pd.to_numeric(df_final[bmi_col], errors="coerce")
    bmi_category = pd.Series(pd.NA, index=df_final.index, dtype="string")
    bmi_category.loc[bmi < 18.5] = "underweight"
    bmi_category.loc[(bmi >= 18.5) & (bmi < 25)] = "normal"
    bmi_category.loc[(bmi >= 25) & (bmi < 30)] = "overweight"
    bmi_category.loc[bmi >= 30] = "obesity"
    df_final["bmi_category"] = bmi_category
    eda_feature_rows.append({
        "feature": "bmi_category", "source": bmi_col, "status": "created",
        "n_positive": np.nan
    })
else:
    eda_feature_rows.append({"feature": "bmi_category", "source": None, "status": "source_not_found", "n_positive": np.nan})


# ----------------------------------------------------------
# High ASA: ASA >= 3
# ----------------------------------------------------------
asa_col = resolve_first_existing_column(df_final, ASA_SOURCE_CANDIDATES)
if asa_col:
    asa_raw = (
        df_final[asa_col]
        .astype("string")
        .str.extract(r"([1-6])", expand=False)
    )
    asa_num = pd.to_numeric(asa_raw, errors="coerce")
    df_final["high_asa"] = pd.Series(pd.NA, index=df_final.index, dtype="Int64")
    valid = asa_num.notna()
    df_final.loc[valid, "high_asa"] = (asa_num.loc[valid] >= 3).astype("Int64")
    eda_feature_rows.append({
        "feature": "high_asa", "source": asa_col, "status": "created",
        "n_positive": int((df_final["high_asa"] == 1).sum())
    })
else:
    eda_feature_rows.append({"feature": "high_asa", "source": None, "status": "source_not_found", "n_positive": 0})

# ----------------------------------------------------------
# Mental Health flag
# ----------------------------------------------------------

if "preop_mental_health" in df_final.columns:
    mental_health_text = (df_final["preop_mental_health"].astype("string").str.strip())

    df_final["mental_health_issue_flag"] = pd.Series(pd.NA, index=df_final.index, dtype="Int64")

    no_mental_health_issues = (mental_health_text.isna()
                        | mental_health_text.eq("")
                        | mental_health_text.str.lower().eq("no mental health issues"))

    df_final.loc[no_mental_health_issues, "mental_health_issue_flag"] = 0
    df_final.loc[~no_mental_health_issues, "mental_health_issue_flag"] = 1

# ----------------------------------------------------------
# Prior surgery flag
# ----------------------------------------------------------
PRIOR_SURGERY_SOURCE_COL = "past_surgery_first_hospitalization"

if PRIOR_SURGERY_SOURCE_COL in df_final.columns:
    prior_surgery_text = (df_final[PRIOR_SURGERY_SOURCE_COL].astype("string").str.strip())

    df_final["prior_surgery_flag"] = pd.Series(pd.NA, index=df_final.index, dtype="Int64")

    no_prior_surgery = (prior_surgery_text.isna()
                        | prior_surgery_text.eq("")
                        | prior_surgery_text.str.lower().eq("no prior surgery"))

    df_final.loc[no_prior_surgery, "prior_surgery_flag"] = 0
    df_final.loc[~no_prior_surgery, "prior_surgery_flag"] = 1

# ----------------------------------------------------------
# Perioperative invasive-device flags.
# CVC/catheter source dates were already timing-gated in Cleaning.
# Drain presence is an intra-operative decision retained from Cleaning.
# ----------------------------------------------------------
device_source_candidates = {
    "has_cvc": ["CVC_start_date"],
    "has_catheter": ["catheter_start_date"],
    "has_drain": DRAIN_PRESENCE_CANDIDATES,
}

device_feature_rows = []

for feature_name, candidates in device_source_candidates.items():
    source_col = resolve_first_existing_column(df_final, candidates)

    if source_col is None:
        device_feature_rows.append({
            "feature": feature_name, "source_column": None,
            "status": "source_column_not_found", "n_positive": 0,
        })
        continue

    s = df_final[source_col]

    if feature_name in {"has_cvc", "has_catheter"}:
        # Existence of a timing-cleaned start date = device was present in the allowed window.
        flag = s.notna().astype("Int64")
    else:
        # Drain field may be binary, YES/NO, Hebrew text, or a non-empty documented value.
        norm = s.astype("string").str.strip().str.lower()
        explicit_no = norm.isin({
            "0", "0.0", "no", "false", "לא", "ללא", "none", "no drain", "ללא נקז"
        })
        explicit_yes = norm.isin({
            "1", "1.0", "yes", "true", "כן", "יש", "drain", "נקז"
        })

        flag = pd.Series(pd.NA, index=df_final.index, dtype="Int64")
        flag.loc[explicit_no] = 0
        flag.loc[explicit_yes] = 1

        # If a non-missing value exists but is not one of the known explicit forms,
        # treat it as documented drain presence; missing stays missing for review.
        documented_other = s.notna() & ~explicit_no & ~explicit_yes & norm.ne("")
        flag.loc[documented_other] = 1

    df_final[feature_name] = flag
    device_feature_rows.append({
        "feature": feature_name,
        "source_column": source_col,
        "status": "created",
        "n_positive": int((flag == 1).sum()),
    })

device_feature_audit = pd.DataFrame(device_feature_rows)

available_device_flags = [
    c for c in ["has_cvc", "has_catheter", "has_drain"]
    if c in df_final.columns
]

if available_device_flags:
    # Require at least one observed device flag; otherwise leave count missing.
    observed_any = df_final[available_device_flags].notna().any(axis=1)
    device_count = df_final[available_device_flags].fillna(0).sum(axis=1).astype("Int64")
    df_final["invasive_device_count"] = pd.Series(pd.NA, index=df_final.index, dtype="Int64")
    df_final.loc[observed_any, "invasive_device_count"] = device_count.loc[observed_any]

# ----------------------------------------------------------
# Surgery bed duration
# ----------------------------------------------------------
SURGERY_BED_START_COL = "surgery_bed_transfer_start_date"
SURGERY_BED_END_COL = "surgery_bed_transfer_end_date"

if (SURGERY_BED_START_COL in df_final.columns 
    and SURGERY_BED_END_COL in df_final.columns):

    surgery_start =  pd.to_datetime(df_final[SURGERY_BED_START_COL], errors="coerce")
    surgery_end =  pd.to_datetime(df_final[SURGERY_BED_END_COL], errors="coerce")

    surgery_duration_minutes = (surgery_end - surgery_start).dt.total_seconds() / 60
    surgery_duration_minutes = surgery_duration_minutes.mask(surgery_duration_minutes < 0)

    df_final["surgery_duration_minutes"] = surgery_duration_minutes

eda_informed_feature_audit = pd.DataFrame(eda_feature_rows)

print("EDA-informed feature audit:")
display(eda_informed_feature_audit)

print("Device feature audit:")
display(device_feature_audit)


# ----------------------------------------------------------
# Recovery duration
# ----------------------------------------------------------
RECOVERY_START_COL = "recovery_bed_transfer_start_date"
RECOVERY_END_COL = "recovery_bed_transfer_end_date"

if (RECOVERY_START_COL in df_final.columns 
    and RECOVERY_END_COL in df_final.columns):

    recovery_start =  pd.to_datetime(df_final[RECOVERY_START_COL], errors="coerce")
    recovery_end =  pd.to_datetime(df_final[RECOVERY_END_COL], errors="coerce")

    recovery_duration_minutes = (recovery_end - recovery_start).dt.total_seconds() / 60
    recovery_duration_minutes = recovery_duration_minutes.mask(recovery_duration_minutes < 0)

    df_final["recovery_duration_minutes"] = recovery_duration_minutes

eda_informed_feature_audit = pd.DataFrame(eda_feature_rows)

print("EDA-informed feature audit:")
display(eda_informed_feature_audit)

print("Device feature audit:")
display(device_feature_audit)


EDA-informed feature audit:


,feature,source,status,n_positive
0,bmi_category,preop_bmi_value,created,NaN
1,high_asa,preop_asa_value,created,5665.0


Device feature audit:


,feature,source_column,status,n_positive
0,has_cvc,CVC_start_date,created,2187
1,has_catheter,catheter_start_date,created,4294
2,has_drain,added1_drain_left_in_place_clean_from_surgery,created,1689


EDA-informed feature audit:


,feature,source,status,n_positive
0,bmi_category,preop_bmi_value,created,NaN
1,high_asa,preop_asa_value,created,5665.0


Device feature audit:


,feature,source_column,status,n_positive
0,has_cvc,CVC_start_date,created,2187
1,has_catheter,catheter_start_date,created,4294
2,has_drain,added1_drain_left_in_place_clean_from_surgery,created,1689


## Step 4 - ICD-9 clinical features and diagnosis/procedure burden

The Cleaning notebook has already reconciled paired text/ICD9 fields conservatively.

This notebook now **creates actual model features** from the offline ICD-9 JSON dictionary:

- grouped comorbidity flags from diagnosis ICD-9 prefixes
- grouped surgical/procedure flags from procedure ICD-9 prefixes
- one compact comorbidity-burden count (`comorbidity_count`)

Two compact count features (`diagnosis_count` and `procedure_count`) are retained, while more redundant binary/count variants are deliberately not created.

The JSON is intentionally editable: if the hospital data contains a code not covered by the curated dictionary, add it to the JSON without changing the Feature Engineering logic.

**Note for Step 6:** some of these ICD9-derived flags (e.g. `has_diabetes`, `has_chronic_kidney_disease`, `has_chronic_pulmonary_disease`) likely overlap with existing structured columns already in the cleaned dataset (`preop_diabetes`, `preop_kidney_disease`, `preop_lung_disease`). This is not a bug here - an ICD9 code can catch a diagnosis the structured field missed, and vice versa - but it does mean near-duplicate columns will exist side by side. Flagging explicitly so it isn't missed at the VIF/MI/L1 redundancy check in Step 6, not for any action in this notebook.


In [18]:
icd9_dictionary = load_icd9_dictionary()

def find_diagnosis_procedure_columns(df_input: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df_input.columns:
        key = col.lower()
        if "diagnos" in key or "proced" in key:
            rows.append({
                "column": col,
                "dtype": str(df_input[col].dtype),
                "n_unique_nonmissing": int(df_input[col].nunique(dropna=True)),
                "missing_percent": float(df_input[col].isna().mean() * 100),
            })
    return pd.DataFrame(rows)


def code_series_normalized(series: pd.Series, kind: str) -> pd.Series:
    return series.map(lambda x: normalize_icd9_code(x, kind))


def build_grouped_flags_from_dictionary(
    df_input: pd.DataFrame,
    dictionary: Dict,
    kind: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    result = df_input.copy()

    if kind == "diagnosis":
        groups = dictionary.get("diagnosis_groups", {})
        candidate_cols = ICD9_DIAGNOSIS_SOURCE_COLUMNS
    elif kind == "procedure":
        groups = dictionary.get("procedure_groups", {})
        candidate_cols = ICD9_PROCEDURE_SOURCE_COLUMNS
    else:
        raise ValueError("kind must be 'diagnosis' or 'procedure'")

    present_cols = [c for c in candidate_cols if c in result.columns]
    rows = []
    created_features = []

    if not present_cols:
        return result, pd.DataFrame([{
            "kind": kind,
            "status": "no_source_columns_found",
            "candidate_columns": " | ".join(candidate_cols),
        }]), created_features

    normalized = {
        c: code_series_normalized(result[c], kind).str.replace(".", "", regex=False)
        for c in present_cols
    }

    for feature_name, spec in groups.items():
        prefixes = [
            str(p).upper().replace(".", "").strip()
            for p in spec.get("prefixes", [])
            if str(p).strip()
        ]

        flag = pd.Series(False, index=result.index)

        for col in present_cols:
            codes = normalized[col]
            for prefix in prefixes:
                flag = flag | codes.str.startswith(prefix, na=False)

        result[feature_name] = flag.astype("Int64")
        created_features.append(feature_name)

        rows.append({
            "kind": kind,
            "feature": feature_name,
            "description": spec.get("description", ""),
            "status": "created",
            "n_positive": int(flag.sum()),
            "n_negative": int((~flag).sum()),
            "prefixes": " | ".join(prefixes),
        })

    return result, pd.DataFrame(rows), created_features


def count_present_pairs(
    df_input: pd.DataFrame,
    text_columns: List[str],
    code_columns: List[str],
) -> pd.Series:
    """
    Count diagnosis/procedure slots where either text OR ICD9 is present.
    This reduces under-counting when only one representation is documented.
    """
    present = pd.DataFrame(index=df_input.index)

    for i in range(max(len(text_columns), len(code_columns))):
        text_col = text_columns[i] if i < len(text_columns) else None
        code_col = code_columns[i] if i < len(code_columns) else None

        slot = pd.Series(False, index=df_input.index)

        if text_col and text_col in df_input.columns:
            text_series = df_input[text_col].astype("string").str.strip()
            slot = slot | (
                df_input[text_col].notna()
                & text_series.ne("")
                & ~text_series.str.contains(
                    r"^No additional ", case=False, regex=True, na=False
                )
            )

        if code_col and code_col in df_input.columns:
            code_series = df_input[code_col].astype("string").str.strip()
            slot = slot | (
                df_input[code_col].notna()
                & code_series.ne("")
                & ~code_series.isin(["0", "0.0"])
            )

        present[f"slot_{i+1}"] = slot

    return present.sum(axis=1).astype("Int64")


diagnosis_procedure_review = find_diagnosis_procedure_columns(df_final)
display(diagnosis_procedure_review)

# Actual grouped diagnosis flags from the offline ICD-9 dictionary.
df_final, diagnosis_group_audit, diagnosis_flag_cols = build_grouped_flags_from_dictionary(
    df_final, icd9_dictionary, kind="diagnosis"
)

# Actual grouped procedure flags from the offline ICD-9 dictionary.
df_final, procedure_group_audit, procedure_flag_cols = build_grouped_flags_from_dictionary(
    df_final, icd9_dictionary, kind="procedure"
)


# Compact documentation-burden features retained by project decision.
# These are counts of existing diagnosis/procedure slots after Cleaning reconciliation.
df_final["diagnosis_count"] = count_present_pairs(
    df_final,
    DIAGNOSIS_TEXT_COLUMNS,
    ICD9_DIAGNOSIS_SOURCE_COLUMNS,
)

df_final["procedure_count"] = count_present_pairs(
    df_final,
    PROCEDURE_TEXT_COLUMNS,
    ICD9_PROCEDURE_SOURCE_COLUMNS,
)

# Keep one compact comorbidity-burden feature only.
# We retain diagnosis_count and procedure_count as compact summary features.
# We still avoid the more redundant variants has_additional_diagnosis,
# has_additional_procedure and procedure_group_count.
if diagnosis_flag_cols:
    df_final["comorbidity_count"] = (
        df_final[diagnosis_flag_cols].fillna(0).sum(axis=1).astype("Int64")
    )

print("Diagnosis grouped features:")
display(diagnosis_group_audit)

print("Procedure grouped features:")
display(procedure_group_audit)


,column,dtype,n_unique_nonmissing,missing_percent
0,diagnosis_1,object,1187,8.530880
1,diagnosis_icd9_1,float64,586,8.810324
2,diagnosis_2,object,955,64.545198
3,diagnosis_icd9_2,object,573,0.000000
4,diagnosis_3,object,617,83.905303
5,diagnosis_icd9_3,object,411,0.000000
6,diagnosis_4,object,487,91.949512
7,diagnosis _icd9_4,object,349,0.000000
8,diagnosis_5,object,300,95.864862
9,diagnosis_icd9_5,float64,216,0.000000


Diagnosis grouped features:


""


Procedure grouped features:


""


## Feature engineering- drug family pattern- stage 5

### Preoperative Medication-Derived Features

The preoperative regular-medication field is stored as free text and contains a heterogeneous mixture of trade names, generic drug names, abbreviations, dosage-related terms, and non-medication words. Therefore, medication-related predictors were not generated automatically from all observed tokens.

A frequency review of the medication vocabulary was first performed. The most common recognizable medications were then reviewed manually and mapped into a limited number of clinically interpretable therapeutic groups. Individual trade names belonging to the same therapeutic class were combined into a single binary feature in order to reduce dimensionality, avoid fragmentation caused by different brand names, and preserve clinically meaningful information.

The following binary medication-derived features were selected:

- `preop_statin_use`- indicates documented preoperative use of a statin or statin-containing lipid-lowering medication. Frequently observed examples in the dataset include Lipitor, Litorva, Stator, Crestor, Atorvastatin and Pravalip. Statins have been evaluated in perioperative populations in relation to postoperative infectious outcomes and may also capture clinically relevant cardiovascular and metabolic background. The feature is therefore included as an exploratory preoperative exposure rather than interpreted as an established protective factor.

- `preop_antiplatelet_use`- indicates documented preoperative use of antiplatelet therapy, including medications such as Aspirin, Micropirin, Cartia and Plavix. Antiplatelet therapy is clinically relevant in the perioperative setting because it reflects important cardiovascular comorbidity and affects perioperative management. It is not interpreted as a direct causal predictor of infection, but as an additional marker of baseline cardiovascular burden and treatment exposure.

- `preop_diabetes_medication_use`- indicates documented use of glucose-lowering therapy. Recognized medications in the dataset include Jardiance, Ozempic, Glucomin/Glucomine, Metformin, Xigduo, Trulicity, insulin preparations and other clinically validated antidiabetic agents. Diabetes and impaired glycemic control are established postoperative risk factors; therefore, medication use may provide complementary information regarding the presence, treatment and potential severity of metabolic disease. This feature is retained in addition to the structured diabetes indicator and does not replace it.

- `preop_chronic_steroid_use`- indicates documented preoperative use of a systemic corticosteroid that may represent chronic immunosuppressive exposure. The mapping includes systemic medications such as Prednisone, Prednisolone, Medrol/Methylprednisolone, Dexamethasone/Decadron and Hydrocortisone/Cortef. Chronic systemic corticosteroid exposure is clinically relevant because of its immunosuppressive effects and its association with impaired wound healing and increased postoperative infectious complications. Although the prevalence of these medications in the study population is relatively low, the feature is retained because it represents a biologically plausible and clinically important high-risk exposure rather than a frequency-driven predictor. Inhaled, topical and other locally administered corticosteroids are not included in this flag unless systemic exposure can be established reliably.

Each medication-family feature is encoded as `1` when at least one validated medication belonging to the corresponding family is identified in the preoperative medication field, and `0` otherwise.

Only explicitly reviewed medication names are included in the mapping. Ambiguous tokens, measurement units, general text terms and uncertain abbreviations are excluded rather than automatically classified.

In [19]:
def normalize_medication_text(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .fillna("")
        .str.upper()
        .str.replace(r"[\r\n\t]+", " ", regex=True)
        .str.replace(r"[,;|/\\]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def extract_medication_candidate_tokens(
    df_input: pd.DataFrame,
    source_col: str = DRUG_SOURCE_COLUMN,
    min_token_length: int = 3,
) -> pd.DataFrame:
    """
    INFO:
    Creates a REVIEW vocabulary, not model features.

    Because medication strings are semi-structured and may contain concatenated words,
    doses and instructions, this tokenization is intentionally conservative.
    Clinical mapping is still required before feature creation.
    """
    if source_col not in df_input.columns:
        return pd.DataFrame(columns=["token", "count", "percent_of_rows"])

    text = normalize_medication_text(df_input[source_col])

    counter = Counter()
    row_count_by_token = Counter()

    token_regex = re.compile(r"[A-Z][A-Z\-]{2,}|[\u0590-\u05FF]{3,}")

    for value in text:
        tokens = token_regex.findall(value)
        cleaned = []

        for token in tokens:
            token = token.strip("-")
            if len(token) < min_token_length:
                continue
            if token in MEDICATION_STOPWORDS:
                continue
            if token.isdigit():
                continue
            cleaned.append(token)

        counter.update(cleaned)
        row_count_by_token.update(set(cleaned))

    n_rows = max(len(df_input), 1)

    rows = [
        {
            "token": token,
            "count": count,
            "rows_containing_token": row_count_by_token[token],
            "percent_of_rows": row_count_by_token[token] / n_rows * 100,
        }
        for token, count in counter.most_common()
    ]

    return pd.DataFrame(rows)

DRUG_FAMILY_PATTERNS: Dict[str, str] = {

    # ----------------------------------------------------------
    # Statins / statin-containing lipid-lowering therapy
    # ----------------------------------------------------------
    "preop_statin_use": (
        r"\b(?:"
        r"LIPITOR|"
        r"LITORVA|"
        r"STATOR|"
        r"CRESTOR|"
        r"ATORVASTATIN|"
        r"PRAVALIP|"
        r"ATOZET"
        r")\b"
    ), 

    # ----------------------------------------------------------
    # Antiplatelet therapy
    # ----------------------------------------------------------
    "preop_antiplatelet_use": (
        r"\b(?:"
        r"ASPIRIN|"
        r"MICROPIRIN|"
        r"CARTIA|"
        r"PLAVIX"
        r")\b"
    ),

    # ----------------------------------------------------------
    # Diabetes medications
    # ----------------------------------------------------------
    "preop_diabetes_medication_use": (
        r"\b(?:"
        r"JARDIANCE|"
        r"OZEMPIC|"
        r"GLUCOMIN|"
        r"GLUCOMINE|"
        r"METFORMIN|"
        r"XIGDUO|"
        r"TRULICITY|"
        r"INSULIN|"
        r"TOUJEO|"
        r"TREGULDEC"
        r")\b"
    ),

    # ----------------------------------------------------------
    # Chronic systemic corticosteroid exposure
    # ----------------------------------------------------------
    "preop_chronic_steroid_use": (
        r"\b(?:"
        r"PREDNISONE|"
        r"PREDNISOLONE|"
        r"MEDROL|"
        r"METHYLPREDNISOLONE|"
        r"DEXAMETHASONE|"
        r"DECADRON|"
        r"HYDROCORTISONE|"
        r"CORTEF"
        r")\b"
    ),
}

def build_medication_family_flags(
    df_input: pd.DataFrame,
    patterns: Dict[str, str],
    source_col: str = DRUG_SOURCE_COLUMN,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    result = df_input.copy()

    if not patterns:
        return result, pd.DataFrame([{
            "status": "not_run_no_clinical_mapping",
            "detail": "DRUG_FAMILY_PATTERNS is empty.",
        }])

    if source_col not in result.columns:
        return result, pd.DataFrame([{
            "status": "source_column_not_found",
            "detail": source_col,
        }])

    text = normalize_medication_text(result[source_col])
    rows = []

    for feature_name, pattern in patterns.items():
        flag = text.str.contains(pattern, case=False, regex=True, na=False)
        result[feature_name] = np.where(flag, "YES", "NO")

        rows.append({
            "feature": feature_name,
            "status": "created",
            "n_positive": int(flag.sum()),
            "n_negative": int((~flag).sum()),
            "regex_pattern": pattern,
        })

    return result, pd.DataFrame(rows)


medication_vocabulary_review = extract_medication_candidate_tokens(df_final)

print("Most frequent candidate medication tokens (review only):")
display(medication_vocabulary_review.head(100))

if ENABLE_DRUG_FLAGS:
    df_final, medication_feature_audit = build_medication_family_flags(
        df_final,
        DRUG_FAMILY_PATTERNS,
    )
else:
    medication_feature_audit = pd.DataFrame([{
        "status": "disabled_until_clinical_drug_family_mapping_is_approved"
    }])

print("Medication-family feature status:")
display(medication_feature_audit)


MEDICATION_TOKENS_OUTPUT_PATH = Path(r"H:\27 project\feature_engineering\fe_outputs\medication_candidate_tokens.xlsx")
medication_vocabulary_review.to_excel(MEDICATION_TOKENS_OUTPUT_PATH, index=False)
print(f"full medication-toke table saved to \n"
      f"{MEDICATION_TOKENS_OUTPUT_PATH}")



Most frequent candidate medication tokens (review only):


,token,count,rows_containing_token,percent_of_rows
0,MGX,7599,2320,7.284373
1,CARDILOC,2850,2812,8.829163
2,MOR,2257,1047,3.287387
3,EVN,1854,896,2.813275
4,NEXIUM,1742,1735,5.447581
...,...,...,...,...
95,DIOVAN,136,133,0.417596
96,CONTROLOC,135,135,0.423875
97,NORMITEN,134,134,0.420735
98,AMBIEN,133,133,0.417596


Medication-family feature status:


,feature,status,n_positive,n_negative,regex_pattern
0,preop_statin_use,created,5907,25942,\b(?:LIPITOR|LITORVA|STATOR|CRESTOR|ATORVASTATIN|PRAVALIP|ATOZET)\b
1,preop_antiplatelet_use,created,3421,28428,\b(?:ASPIRIN|MICROPIRIN|CARTIA|PLAVIX)\b
2,preop_diabetes_medication_use,created,2747,29102,\b(?:JARDIANCE|OZEMPIC|GLUCOMIN|GLUCOMINE|METFORMIN|XIGDUO|TRULICITY|INSULIN|TOUJEO|TREGULDEC)\b
3,preop_chronic_steroid_use,created,130,31719,\b(?:PREDNISONE|PREDNISOLONE|MEDROL|METHYLPREDNISOLONE|DEXAMETHASONE|DECADRON|HYDROCORTISONE|CORTEF)\b


full medication-toke table saved to 
H:\27 project\feature_engineering\fe_outputs\medication_candidate_tokens.xlsx


## Step 6 - Surgeon and Anesthesiologist Code Audit

The clinician-code audit was performed to assess whether physician identifiers are suitable for direct inclusion as model predictors.

For `surgeon_code`, the audit identified a high-cardinality categorical variable with 371 unique non-missing surgeon codes across the analyzed population. In addition, a substantial number of surgeon codes were rare: 129 codes appeared in fewer than five surgical cases.

These findings indicate that the raw surgeon identifier is not an optimal predictor for direct model inclusion. A high-cardinality identifier can generate a large number of sparse categories, particularly when many clinicians are represented by only a few cases. Such sparse categories provide limited statistical support for learning stable associations and may increase the risk of overfitting.

Furthermore, `surgeon_code` represents the identity of a specific clinician rather than an intrinsic clinical characteristic of the patient or procedure. The model could therefore learn local patterns related to physician identity, referral patterns, case mix, or the types of procedures usually performed by a specific surgeon, rather than learning generalizable clinical risk factors. This may reduce model transportability when applied to future cases, new clinicians, or other clinical settings.

The same methodological considerations apply to `anesthesiologist_code`. 

Accordingly, raw clinician identifiers are not selected for direct model inclusion. More generalizable variables, such as surgical specialty, procedure group, and other clinical or procedural characteristics, are preferred because they retain relevant clinical information without encoding the identity of an individual clinician.

In [20]:
def clinician_code_audit(
    df_input: pd.DataFrame,
    columns: List[str] = CLINICIAN_CODE_COLS,
    rare_count_threshold: int = 5,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    frequency_rows = []

    for col in columns:
        if col not in df_input.columns:
            summary_rows.append({
                "column": col,
                "status": "column_not_found",
            })
            continue

        counts = df_input[col].value_counts(dropna=False)
        nonmissing = df_input[col].dropna()

        rare_codes = counts[counts < rare_count_threshold]

        summary_rows.append({
            "column": col,
            "status": "audited",
            "n_rows": len(df_input),
            "n_unique_nonmissing": int(nonmissing.nunique()),
            "missing_percent": float(df_input[col].isna().mean() * 100),
            "n_codes_with_fewer_than_5_rows": int(len(rare_codes)),
            "rows_in_codes_with_fewer_than_5_rows": int(rare_codes.sum()),
        })

        for code_value, count in counts.items():
            frequency_rows.append({
                "column": col,
                "code": code_value,
                "count": int(count),
                "percent": float(count / len(df_input) * 100),
            })

    return pd.DataFrame(summary_rows), pd.DataFrame(frequency_rows)


clinician_summary_audit, clinician_frequency_audit = clinician_code_audit(df_final)

print("Clinician-code summary:")
display(clinician_summary_audit)

print("Most frequent clinician codes:")
display(
    clinician_frequency_audit
    .sort_values(["column", "count"], ascending=[True, False])
    .groupby("column", group_keys=False)
    .head(20)
)

clinician_id_cols_to_drop = [col for col in CLINICIAN_CODE_COLS
                            if col in df_final.columns]
if clinician_id_cols_to_drop:
    df_final = df_final.drop(columns=clinician_id_cols_to_drop)

print("Raw clinician identifier columns excluded from model features:")
print(clinician_code_audit)

Clinician-code summary:


,column,status,n_rows,n_unique_nonmissing,missing_percent,n_codes_with_fewer_than_5_rows,rows_in_codes_with_fewer_than_5_rows
0,surgeon_code,audited,31849,371,0.021979,129,249
1,anasthesiologist_code,audited,31849,190,1.742598,90,154


Most frequent clinician codes:


,column,code,count,percent
372,anasthesiologist_code,1098979.0,1222,3.836855
373,anasthesiologist_code,8520069.0,1112,3.491475
374,anasthesiologist_code,4611990.0,967,3.036202
375,anasthesiologist_code,2062393.0,960,3.014223
376,anasthesiologist_code,6368250.0,900,2.825834
377,anasthesiologist_code,8955606.0,757,2.376841
378,anasthesiologist_code,6693560.0,745,2.339163
379,anasthesiologist_code,1712430.0,744,2.336023
380,anasthesiologist_code,3049589.0,710,2.229269
381,anasthesiologist_code,9594205.0,697,2.188452


Raw clinician identifier columns excluded from model features:
<function clinician_code_audit at 0x00000240813D9D00>


In [21]:
# Redundancy check: imaging related raw columns
from scipy.stats import chi2_contingency

ct = pd.crosstab(df_final["added1_imaging_test"], df_final["preop_chest_imaging_normality"])
chi2, p, dof, _ = chi2_contingency(ct)
n = ct.sum().sum()
cramers_v = (chi2 / (n * (min(ct.shape)-1))) ** 0.5

display(ct)
print(f"p-value: {p:.4f}, Cramer's V: {cramers_v:.3f}")

preop_chest_imaging_normality,לא אבחנתי,לא תקין,תקין
added1_imaging_test,,,
אגן ירכיים 2 צילומים,0,2,19
אגן עצמות-mri,0,0,2
ברך 2 מבטים צילום,2,1,35
ברך-mri,0,0,5
"ע""ש גבי (ct)",0,0,1
"ע""ש גבי-mri",0,0,4
"ע""ש מותני (ct)",3,0,22
"ע""ש מותני-mri",0,2,29
"ע""ש צווארי (ct)",0,0,5


p-value: 0.9991, Cramer's V: 0.220


## Step 7- Raw-source cleanup for the final tabular feature table

This step is intentionally narrow.

### NLP
After the nlp-derived `risk_score` has been merged successfully, the original free-text NLP narratives are **not** used directly as tabular model predictors.


### ICD9 / DIAGNOSIS / PROCEDURES
Raw ICD9 and redundant text descriptions are removed only after dictionary-driven diagnosis/procedure features were successfully created. The raw source fields are saved in the reference table.

### Medication text
`preop_regular_medications` is removed **only if** approved medication-family features were actually created.

This prevents accidental loss of information before its derived features exist.


In [ ]:
def cleanup_raw_source_fields(
    df_input: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    result = df_input.copy()
    audit_rows = []
    reference_cols = []

    # ----------------------------------------------------------
    # Admission/surgery/discharge date columns:
    # Used transiently in Step 3 to compute transferred_to_icu_directly - always
    # dropped once that is done, unconditionally (no ENABLE_ flag - there is no safe
    # scenario for a raw timestamp to reach the model as a predictor).
    # ----------------------------------------------------------
    for col in DATE_SOURCE_COLS_TO_DROP:
        if col in result.columns:
            reference_cols.append(col)
            result = result.drop(columns=col)
            audit_rows.append({
                "column": col,
                "action": "removed_from_tabular_predictors",
                "reason": "raw timestamp; not a model feature itself",
            })

    # ----------------------------------------------------------
    # EDA Decision:
    # admission_year, home_town
    # removing payment_by because we created "payment_category" column
    # ----------------------------------------------------------
    EDA_COLS_TO_DROP = ["admission_year", "home_town", "payment _by"]
    for col in EDA_COLS_TO_DROP:
        if col in result.columns:
            result = result.drop(columns=col)
            audit_rows.append({
                "column": col,
                "action": "removed_after_clean_eda_decision",
                "reason": "no important information",
            })

    # ----------------------------------------------------------
    # Raw source columns used only to derive new features:
    # prior_surgery_flag and recovery_duration_minutes
    # ----------------------------------------------------------
    for col in DERIVED_FEATURE_RAW_COLS_TO_DROP:
        if col in result.columns:
            reference_cols.append(col)
            result = result.drop(columns=col)
            audit_rows.append({
                "column": col,
                "action": "removed_from_tabular_predictors",
                "reason": "raw source column replaced by a derived feature",
            })

    # ----------------------------------------------------------
    # ICU transfer raw source columns:
    # Same reasoning as the date columns above - used only to compute
    # transferred_to_icu_directly in Step 3, always dropped unconditionally once
    # that is done (a raw bed identifier or timestamp is never a safe predictor).
    # ----------------------------------------------------------
    for col in ICU_RAW_COLS_TO_DROP:
        if col in result.columns:
            reference_cols.append(col)
            result = result.drop(columns=col)
            audit_rows.append({
                "column": col,
                "action": "removed_from_tabular_predictors",
                "reason": "raw ICU bed/timestamp; only used to compute transferred_to_icu_directly",
            })

    # ----------------------------------------------------------
    # CVC / catheter / drain raw source columns:
    # Same reasoning as ICU above - has_cvc/has_catheter/has_drain (Step 5) are the
    # intended tabular representation; the raw timing-gated timestamps
    # (CVC_start_date, catheter_start_date) and the raw drain field are dropped once
    # those flags exist, conditional on the flag actually having been created (mirrors
    # the ICD9/medication "approved features created" pattern below).
    # ----------------------------------------------------------
    device_raw_source_map = {
        "has_cvc": "CVC_start_date",
        "has_catheter": "catheter_start_date",
        "has_drain": "added1_drain_left_in_place_clean_from_surgery",
    }
    for flag_col, raw_col in device_raw_source_map.items():
        if flag_col in result.columns and raw_col in result.columns:
            reference_cols.append(raw_col)
            result = result.drop(columns=raw_col)
            audit_rows.append({
                "column": raw_col,
                "action": "removed_after_feature_extraction",
                "reason": f"raw source for {flag_col}; not a model feature itself",
            })

    # ----------------------------------------------------------
    # ICD9 + confirmed redundant diagnosis/procedure descriptions:
    # Drop only if approved grouped flags were actually created.
    # ----------------------------------------------------------
    approved_icd_features_created = bool(diagnosis_flag_cols or procedure_flag_cols)

    if approved_icd_features_created:
        for col in (
            PROCEDURE_TEXT_COLUMNS
            + DIAGNOSIS_TEXT_COLUMNS
            + CONFIRMED_REDUNDANT_DIAGNOSIS_PROCEDURE_TEXT_COLS
        ):
            if col in result.columns:
                reference_cols.append(col)
                result = result.drop(columns=col)
                audit_rows.append({
                    "column": col,
                    "action": "removed_after_feature_extraction",
                    "reason": "raw/redundant ICD9 diagnosis/procedure representation",
                })

    # ----------------------------------------------------------
    # NLP raw narratives:
    # Move to reference, but do not use as direct tabular predictors.
    # ----------------------------------------------------------
    for col in NLP_RAW_TEXT_COLS:
        if col in result.columns:
            reference_cols.append(col)
            result = result.drop(columns=col)
            audit_rows.append({
                "column": col,
                "action": "removed_from_tabular_predictors",
                "reason": "raw NLP narrative; derived NLP features are the intended tabular representation",
            })

    # ----------------------------------------------------------
    # Medication free text:
    # Drop only after approved medication-family features exist.
    # ----------------------------------------------------------
    approved_drug_features_created = (
        ENABLE_DRUG_FLAGS
        and bool(DRUG_FAMILY_PATTERNS)
        and all(feature in result.columns for feature in DRUG_FAMILY_PATTERNS)
    )

    if approved_drug_features_created and DRUG_SOURCE_COLUMN in result.columns:
        reference_cols.append(DRUG_SOURCE_COLUMN)
        result = result.drop(columns=[DRUG_SOURCE_COLUMN])
        audit_rows.append({
            "column": DRUG_SOURCE_COLUMN,
            "action": "removed_after_feature_extraction",
            "reason": "raw multi-medication text replaced by approved medication-family features",
        })

    reference_cols = list(dict.fromkeys(reference_cols))
    reference_table_cols = [CASE_ID_COL] + [
        c for c in reference_cols if c in df_input.columns
    ]

    reference_table = df_input[reference_table_cols].copy()

    return result, pd.DataFrame(audit_rows), reference_table


final_feature_table, raw_source_cleanup_audit, raw_source_reference = cleanup_raw_source_fields(df_final)

print("Raw-source cleanup audit:")
display(raw_source_cleanup_audit)


Raw-source cleanup audit:


,column,action,reason
0,admission_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
1,activity_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
2,icu_transfer_start_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
3,surgery_bed_transfer_start_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
4,surgey_bed_transfer_end_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
5,recovery_bed_transfer_start_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
6,recovery_bed_transfer_end_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
7,CVC_start_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
8,catheter_start_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself
9,added1_drain_left_in_place_start_date,removed_from_tabular_predictors,raw timestamp; not a model feature itself


## Step 9 - Final Feature Engineering audit

This audit documents what was created, reviewed and removed.  
It does not perform feature selection.


In [23]:

feature_creation_summary = pd.DataFrame([
    {
        "component": "NLP merge",
        "status": "enabled" if USE_NLP_FEATURES else "disabled / waiting for NLP feature file",
    },
    {
        "component": "transferred_to_icu_directly",
        "status": "created; missing/no ICU transfer record is clinically allowed and encoded as 0 for this derived flag"
        if "transferred_to_icu_directly" in final_feature_table.columns else "not created",
    },
    {
        "component": "ICD9 diagnosis group flags",
        "status": f"{len(diagnosis_flag_cols)} feature(s) created from offline JSON",
    },
    {
        "component": "ICD9 procedure group flags",
        "status": f"{len(procedure_flag_cols)} feature(s) created from offline JSON",
    },
    {
        "component": "Comorbidity burden",
        "status": "comorbidity_count created" if "comorbidity_count" in final_feature_table.columns else "not created",
    },
    {
        "component": "EDA-informed baseline features",
        "status": "bmi_category / high_asa created when source columns exist",
    },
    {
        "component": "Perioperative device features",
        "status": "has_cvc / has_catheter / has_drain / invasive_device_count created when source columns exist",
    },
   {
    "component": "Medication-family features",
    "status": (
        "created: preop_statin_use / preop_antiplatelet_use / "
        "preop_diabetes_medication_use / preop_chronic_steroid_use"
        if ENABLE_DRUG_FLAGS
        and all(
            feature in final_feature_table.columns
            for feature in DRUG_FAMILY_PATTERNS
        )
        else "not created"
            ),
    },
    {
        "component": "Final cohort definition / undersampling",
        "status": "NOT performed here; fixed cohort restrictions are applied pre-split in Step 6, prototype undersampling is Train-only after split",
    },
    {
        "component": "Train/Test split",
        "status": "NOT performed in this notebook",
    },
    {
        "component": "One-Hot / distribution-dependent imputation / VIF / feature selection",
        "status": "NOT performed in this notebook",
    },
    {
    "component": "Prior surgery feature",
    "status": (
        "prior_surgery_flag created from past_surgery_first_hopitalization"
        if "prior_surgery_flag" in final_feature_table.columns
        else "not created"
            ),
    },
    {
        "component": "Recovery duration feature",
        "status": (
            "recovery_duration_minutes created from recovery start/end timestamps"
            if "recovery_duration_minutes" in final_feature_table.columns
            else "not created"
        ),
    },
    {
        "component": "Clinician identifiers",
        "status": (
            "raw surgeon_code excluded after high-cardinality audit; "
            "clinician identifiers are not used as direct model predictors"
        ),
    },

])

display(feature_creation_summary)

print(f"Input cleaned table shape: {df_final.shape}")
print(f"Final Feature Engineering table shape: {final_feature_table.shape}")


,component,status
0,NLP merge,enabled
1,transferred_to_icu_directly,created; missing/no ICU transfer record is clinically allowed and encoded as 0 for this derived flag
2,ICD9 diagnosis group flags,0 feature(s) created from offline JSON
3,ICD9 procedure group flags,0 feature(s) created from offline JSON
4,Comorbidity burden,not created
5,EDA-informed baseline features,bmi_category / high_asa created when source columns exist
6,Perioperative device features,has_cvc / has_catheter / has_drain / invasive_device_count created when source columns exist
7,Medication-family features,created: preop_statin_use / preop_antiplatelet_use / preop_diabetes_medication_use / preop_chronic_steroid_use
8,Final cohort definition / undersampling,"NOT performed here; fixed cohort restrictions are applied pre-split in Step 6, prototype undersampling is Train-only after split"
9,Train/Test split,NOT performed in this notebook


Input cleaned table shape: (31849, 112)
Final Feature Engineering table shape: (31849, 92)


## Step 10 - Save the FINAL FEATURE TABLE and audits


In [24]:
FINAL_FEATURE_TABLE_PATH = OUTPUT_DIR / "assuta_final_feature_table.csv"
REFERENCE_TABLE_PATH = OUTPUT_DIR / "feature_engineering_raw_source_reference.csv"
AUDIT_PATH = OUTPUT_DIR / "feature_engineering_audit.xlsx"

final_feature_table.to_csv(
    FINAL_FEATURE_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)

raw_source_reference.to_csv(
    REFERENCE_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)

audit_tables = {
    "feature_summary": feature_creation_summary,
    #"nlp_merge": nlp_merge_audit,
    #"icu_transfer_feature": icu_transfer_audit,
    "diagnosis_procedure_review": diagnosis_procedure_review,
    "icd9_diagnosis_features": diagnosis_group_audit,
    "icd9_procedure_features": procedure_group_audit,
    "device_features": device_feature_audit,
    "eda_informed_features": eda_informed_feature_audit,
    "medication_vocabulary": medication_vocabulary_review,
    "medication_features": medication_feature_audit,
    "clinician_summary": clinician_summary_audit,
    "clinician_frequency": clinician_frequency_audit,
    "raw_source_cleanup": raw_source_cleanup_audit,
    "final_columns": pd.DataFrame({"column": final_feature_table.columns}),
}

with pd.ExcelWriter(AUDIT_PATH, engine="openpyxl") as writer:
    for sheet_name, table in audit_tables.items():
        if isinstance(table, pd.DataFrame):
            table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"Saved FINAL FEATURE TABLE: {FINAL_FEATURE_TABLE_PATH}")
print(f"Saved raw-source reference table: {REFERENCE_TABLE_PATH}")
print(f"Saved Feature Engineering audit workbook: {AUDIT_PATH}")


Saved FINAL FEATURE TABLE: H:\27 project\feature_engineering\fe_outputs\assuta_final_feature_table.csv
Saved raw-source reference table: H:\27 project\feature_engineering\fe_outputs\feature_engineering_raw_source_reference.csv
Saved Feature Engineering audit workbook: H:\27 project\feature_engineering\fe_outputs\feature_engineering_audit.xlsx


## Step 11 - Final validation gate

This gate checks that the Feature Engineering handoff is structurally safe.

It does **not** require all features to be numeric and does **not** require missing-value imputation, because those belong to the next stage after Train/Test split.


**Final cohort restrictions are applied in Step 6 before the Train/Test split; prototype/aggregation undersampling is applied only to Train after the split. Neither is performed in Feature Engineering.**


In [25]:
print("=" * 72)
print("FINAL FEATURE ENGINEERING VALIDATION GATE")
print("=" * 72)

ok = True

if TARGET_COL not in final_feature_table.columns:
    ok = False
    print(f"FAIL: {TARGET_COL} is missing.")
else:
    print(f"OK: {TARGET_COL} is present.")

if CASE_ID_COL not in final_feature_table.columns:
    ok = False
    print(f"FAIL: {CASE_ID_COL} is missing.")
else:
    print(f"OK: {CASE_ID_COL} is retained as the surgical-case reference key.")

if final_feature_table[CASE_ID_COL].duplicated().any():
    ok = False
    print("FAIL: duplicate surgical case IDs exist in final feature table.")
else:
    print("OK: one row per surgical case.")

raw_nlp_left = [c for c in NLP_RAW_TEXT_COLS if c in final_feature_table.columns]
if raw_nlp_left:
    ok = False
    print("FAIL: raw NLP narrative columns remain in tabular predictors:", raw_nlp_left)
else:
    print("OK: raw NLP narratives are not direct tabular predictors.")



required_engineered_features = [
    "transferred_to_icu_directly",
]

missing_engineered = [
    c for c in required_engineered_features
    if c not in final_feature_table.columns
]

if missing_engineered:
    ok = False
    print("FAIL: required engineered features missing:", missing_engineered)

else:
    print("OK: core deterministic engineered features are present.")

redundant_features_that_should_not_exist = [
    "has_additional_diagnosis",
    "has_additional_procedure",
    "procedure_group_count",
    "received_preop_antibiotic",
    "preop_antibiotic_type",
    "age_65_plus",
    "cardiothoracic_surgery",
    "spine_surgery",
    "high_risk_monday_or_thursday",
]

unexpected_redundant = [
    c for c in redundant_features_that_should_not_exist
    if c in final_feature_table.columns
]

if unexpected_redundant:
    ok = False
    print("FAIL: intentionally excluded/redundant engineered features are present:", unexpected_redundant)
else:
    print("OK: intentionally excluded/redundant engineered features were not created.")

if len(final_feature_table) != len(df_final):
    ok = False
    print(
        f"FAIL: row count changed during Feature Engineering "
        f"({len(df_final)} -> {len(final_feature_table)})."
    )
else:
    print("OK: Feature Engineering did not change the number of surgical cases.")

print()
if ok:
    print("READY FOR STEP 6: SPLIT + TRAIN-LEARNED PREPROCESSING + FEATURE SELECTION.")
else:
    print("NOT READY — review FAIL item(s) above.")


FINAL FEATURE ENGINEERING VALIDATION GATE
OK: target_binary is present.
OK: case_number is retained as the surgical-case reference key.
OK: one row per surgical case.
OK: raw NLP narratives are not direct tabular predictors.
OK: core deterministic engineered features are present.
OK: intentionally excluded/redundant engineered features were not created.
OK: Feature Engineering did not change the number of surgical cases.

READY FOR STEP 6: SPLIT + TRAIN-LEARNED PREPROCESSING + FEATURE SELECTION.
